In [1]:
%matplotlib widget

In [46]:
from ctypes.wintypes import PUINT
from diagrams import Cluster, Diagram
from diagrams.gcp.analytics import BigQuery, Dataflow, PubSub
from diagrams.gcp.compute import AppEngine, Functions
from diagrams.gcp.database import BigTable
from diagrams.gcp.iot import IotCore
from diagrams.gcp.storage import GCS

with Diagram("GECCO Simulation", show=False, direction="TB"):
    pubsub = Functions("GECCO Initialization")

    with Cluster("GECCO Python Script"):
        [IotCore("Patient CT Volume"),
         IotCore("Spekpy Spectrum"),
         IotCore("CBCT Geometry")] >> pubsub


    
    with Cluster("Simulation Files"):

        with Cluster("GGEMS Specific Files"):
            ggems_sim = AppEngine("Material File")
            pubsub >> ggems_sim
        with Cluster("General files (for both)"):
            general_sim = [GCS("CT mhd"),
            GCS("CT range"),
            GCS("Spectrum dat")]
            with Cluster("Preset Files"):
                GCS("Detector OSF") 
                GCS("Bowtie Filter")
            pubsub >> general_sim
           
        with Cluster("GECCO Specific Files"):
            gecco_sim = [AppEngine("CT density"),
                        AppEngine("Attenuation Data")]
            pubsub >> gecco_sim

    with Cluster("GECCO Simulation"):

        with Cluster("Primary Projections"):
            fc_sim = PubSub("Fastcat Simulation")
            # general_sim >>  fc_sim
            gecco_sim >> fc_sim

        with Cluster("Secondary Projections"):
            gg_sim = PubSub("GGEMS Simulation")
            # general_sim >> gg_sim
            ggems_sim >> gg_sim

    postprocess = Functions("GECCO Postprocessing")
    fc_sim >> postprocess
    gg_sim >> postprocess

    postprocess >> BigTable("GECCO CBCT")
            
            # with Cluster("Processing"):
            #     PubSub("Material File") # >> BigTable("bigtable")

            # with Cluster("Serverless"):
            #     Functions("func") >> AppEngine("appengine")

    # pubsub >> flow